# Fase 1 · extracción
Modelo doble determinista. LCEL, @chain, invoke, batch y stream.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from langchain_core.runnables import RunnableLambda
from pedro.chains import crear_lcel, crear_chain
import json
modelo = RunnableLambda(lambda prompt: json.dumps({'intencion':'comparar',
    'solicitud':{'producto_id':'PEC-01-0001','cantidad':2,'presupuesto_centavos':100000,
    'plazo_max_dias':7},'faltantes':[]}, ensure_ascii=False))
lcel, decorada = crear_lcel(modelo), crear_chain(modelo)
entrada = {'texto':'Compara dos unidades de PEC-01-0001 hasta R$ 1.000 en 7 días'}
print(lcel.invoke(entrada).model_dump_json())
print(decorada.invoke(entrada).model_dump_json())
print([x.model_dump() for x in lcel.batch([entrada, entrada])])
print(list(modelo.stream('etapa del modelo')))
print(lcel.get_graph().draw_mermaid())

{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null,"origen_preferido":null},"faltantes":[]}
{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null,"origen_preferido":null},"faltantes":[]}
[{'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None, 'origen_preferido': None}, 'faltantes': []}, {'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None, 'origen_preferido': None}, 'faltantes': []}]
['{"intencion": "comparar", "solicitud": {"producto_id": "PEC-01-0001", "cantidad": 2, "presupuesto_centavos": 100000, "plazo_max_dias": 7}, "faltantes": []}']
---
config:
  flowcha

LCEL expone etapas declarativas. @chain envuelve la misma extracción. Los faltantes se recalculan tras el parser.

## R03 · plantilla reutilizable con variables del dominio
Ejecución real local, sin modelo: se rellena la misma `ChatPromptTemplate` con tres solicitudes distintas y se muestran los mensajes resultantes. La plantilla vive en `pedro.chains.PROMPT` y la reutilizan la cadena LCEL y la cadena `@chain`. La ejecución con el modelo real está en `docs/trazas/fase1_chain_llm.json`.

In [3]:
from pedro.chains import PROMPT
print('clase:', type(PROMPT).__name__, '| variables:', sorted(PROMPT.input_variables))
print('mensajes:', [type(m).__name__ for m in PROMPT.messages])
parcial = PROMPT.partial(formato='Devuelve JSON con intencion, solicitud y faltantes.')
print('tras partial -> variables libres:', sorted(parcial.input_variables))
for texto in ('Compara 2 unidades de PEC-01-0001 hasta R$ 5.000 en 7 días',
              'Necesito un termostato para nevera, sin prisa',
              '¿A qué ciudad corresponde el CEP 01001000?'):
    sistema, humano = parcial.format_messages(texto=texto)
    print(humano.type, '->', humano.content)
print(sistema.type, '->', sistema.content)
print('misma plantilla en LCEL:', lcel.first.messages == PROMPT.messages)

clase: ChatPromptTemplate | variables: ['formato', 'texto']
mensajes: ['SystemMessagePromptTemplate', 'HumanMessagePromptTemplate']
tras partial -> variables libres: ['texto']
human -> Compara 2 unidades de PEC-01-0001 hasta R$ 5.000 en 7 días
human -> Necesito un termostato para nevera, sin prisa
human -> ¿A qué ciudad corresponde el CEP 01001000?
system -> Persona: Pedro. Tarea: extraer intención y solicitud de compra ficticia. Contexto: ninguna cifra ausente se infiere como cero. Formato: JSON válido. Devuelve JSON con intencion, solicitud y faltantes.
misma plantilla en LCEL: True


## R04 · laboratorio de salidas rotas
Ocho salidas típicas de un modelo pasan por la cadena con reparación (`crear_lcel_robusta`). Cada defecto lo resuelve una capa distinta: el **parser** (tolera cercas y texto alrededor), la **reparación** (otra cadena corrige con la solicitud, la salida inválida y el error) o una **regla de dominio** (los faltantes los calcula el código). **Doble de prueba:** el generador emite la salida rota y el reparador devuelve siempre el JSON correcto; la reparación con el modelo real está en `docs/trazas/fase1_chain_llm.json`.

In [4]:
from pedro.laboratorio import CORRECTA, SALIDAS_ROTAS, SOLICITUD, ensayar
reparador_doble = RunnableLambda(lambda prompt: json.dumps(CORRECTA))
print('solicitud:', SOLICITUD)
for caso in ensayar(reparador_doble):
    print(f"{caso['caso']:28} capa: {caso['capa_observada']:17} reparaciones: {caso['reparaciones']}"
          f" -> {caso['resultado']['solicitud']} faltan {caso['resultado']['faltantes']}")

solicitud: Compara 2 unidades de PEC-01-0001 hasta R$ 5.000 en 7 días
JSON válido                  capa: ninguna           reparaciones: 0 -> {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} faltan []
cercas markdown y charla     capa: parser            reparaciones: 0 -> {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} faltan []
JSON truncado                capa: reparación        reparaciones: 1 -> {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} faltan []
cantidad en letras           capa: reparación        reparaciones: 1 -> {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} faltan []
cantidad negativa            capa: reparación        reparaciones: 1 -> {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} faltan []
intención fuera de la lis

## R04 · un parser tolerante puede perder datos en silencio
Hallazgo del laboratorio. El parser de LangChain acepta JSON cortado. Con la cadena simple, una salida truncada se convierte en un objeto válido al que le falta el plazo, sin ningún error. La cadena robusta comprueba que el JSON esté entero y, si no, lo manda a reparar.

In [5]:
truncada = SALIDAS_ROTAS['JSON truncado'][0]
print('salida del modelo:', truncada)
simple = crear_lcel(RunnableLambda(lambda _: truncada)).invoke({'texto': SOLICITUD})
print('cadena simple  -> plazo:', simple.solicitud.plazo_max_dias, '(se perdió sin error)')
from pedro.chains import crear_lcel_robusta
robusta = crear_lcel_robusta(RunnableLambda(lambda _: truncada), reparador_doble).invoke({'texto': SOLICITUD})
print('cadena robusta -> plazo:', robusta.solicitud.plazo_max_dias, '(detectado y reparado)')

salida del modelo: {"intencion": "comparar", "faltantes": [], "solicitud": {"producto_id": "PEC-01-0001", "cantidad": 2, "presupuesto_centavos": 500000, "plazo_ma
cadena simple  -> plazo: None (se perdió sin error)
cadena robusta -> plazo: 7 (detectado y reparado)


## R05 · invoke, batch y stream sobre una cadena @chain
**Doble de prueba:** el modelo es un generador que entrega el JSON en trozos de 12 caracteres, para que `stream()` tenga algo que transmitir. Se ejecutan los tres métodos sobre la cadena imperativa. `stream()` solo transmite si todos los pasos lo permiten: con el parser Pydantic llega un único fragmento; con el parser de JSON parcial llega el objeto a medida que se forma. Las mediciones con el modelo real están en `docs/trazas/fase1_chain_llm.json`.

In [6]:
from pedro.chains import crear_chain_robusta, crear_lcel_parcial
texto_json = json.dumps(CORRECTA)
def modelo_por_trozos(_):
    for i in range(0, len(texto_json), 12):
        yield texto_json[i:i + 12]
doble = RunnableLambda(modelo_por_trozos)
imperativa = crear_chain(doble)
entrada = {'texto': SOLICITUD}
print('invoke:', imperativa.invoke(entrada).solicitud.model_dump(exclude_none=True))
print('batch :', [r.intencion for r in imperativa.batch([entrada] * 3)])
print('stream con @chain y Pydantic:', len(list(imperativa.stream(entrada))), 'fragmento')
estados = list(crear_lcel_parcial(doble).stream(entrada))
print('stream con JSON parcial:', len(estados), 'estados')
for estado in estados[::3]:
    print('  ', estado)

invoke: {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7}
batch : ['comparar', 'comparar', 'comparar']
stream con @chain y Pydantic: 1 fragmento
stream con JSON parcial: 8 estados
   {}
   {'intencion': 'comparar', 'faltantes': [], 'solicitud': {}}
   {'intencion': 'comparar', 'faltantes': [], 'solicitud': {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000}}


## R05 · la cadena con reparación, escrita con @chain
`crear_chain_robusta` hace lo mismo que la versión declarativa de R04, pero el control de errores es un `try/except` a la vista. Se pasa el mismo laboratorio por las dos versiones.

In [7]:
import inspect
print(inspect.getsource(crear_chain_robusta).split('    @chain\n')[1].split('    return extraer_con_reparacion')[0])
print('imperativa = declarativa en los 8 casos:',
      ensayar(reparador_doble, crear_chain_robusta) == ensayar(reparador_doble))

    def extraer_con_reparacion(dato: dict) -> Extraccion:
        salida = generar.invoke(dato)
        try:
            extraccion = parser.parse(salida)
            if not _json_completo(salida):
                raise OutputParserException("JSON incompleto: la salida se cortó antes de cerrar")
        except OutputParserException as error:
            corregida = reparar.invoke({"texto": dato["texto"], "salida": salida, "error": error})
            extraccion = parser.parse(corregida)
        return _validar(extraccion)


imperativa = declarativa en los 8 casos: True


## R06 · la misma cadena en LCEL y en @chain: qué deja ver cada una
Sin modelo (dobles). Las dos versiones de la cadena con reparación dan el mismo resultado. Aquí se mide lo que cambia: los pasos que registra la traza, si el error queda anotado y si el grafo muestra la reparación.

In [8]:
from pedro.laboratorio import comparar_estilos
medidas = comparar_estilos()
for estilo, dato in medidas.items():
    print(estilo)
    for clave, valor in dato.items():
        if clave != 'pasos_con_reparacion':
            print('  ', clave, '=', valor)
    print('   pasos:', ' > '.join(dato['pasos_con_reparacion']))
print(crear_lcel_robusta(RunnableLambda(lambda _: ''), reparador_doble).get_graph().draw_mermaid())

declarativa (LCEL)
   pasos_trazados_sin_fallo = 10
   pasos_trazados_con_reparacion = 16
   el_error_queda_en_la_traza = True
   la_validacion_de_dominio_queda_en_la_traza = True
   el_grafo_muestra_la_reparacion = True
   resultado_tras_reparar = 2
   pasos: RunnableSequence > RunnableAssign<salida> > RunnableParallel<salida> > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > RunnableWithFallbacks > analizar_estricto > ERROR:OutputParserException > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > PydanticOutputParser > _validar
imperativa (@chain)
   pasos_trazados_sin_fallo = 5
   pasos_trazados_con_reparacion = 9
   el_error_queda_en_la_traza = False
   la_validacion_de_dominio_queda_en_la_traza = False
   el_grafo_muestra_la_reparacion = False
   resultado_tras_reparar = 2
   pasos: extraer_con_reparacion > RunnableSequence > ChatPromptTemplate > RunnableLambda > StrOutputParser > RunnableSequence > ChatPromptTemplate > RunnableL

Lectura: en LCEL la traza registra el fallo del parser, la reparación y la validación de dominio como pasos con nombre, y el grafo muestra el `WithFallbacks`. En `@chain` el resultado es idéntico, pero el `try/except` y la llamada a `_validar` son Python normal: la traza no los ve.

## R01 · anatomía del agente
La anatomía se lee de los objetos reales del agente final (`pedro.anatomia`), no se escribe a mano. No se llama a ningún modelo: el executor se construye con un modelo de relleno que nunca se invoca.

In [9]:
import json
from pedro.catalog import crear_escenario
from pedro.anatomia import describir_anatomia
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
memoria = Path('storage/notebooks/r01.sqlite3')
anatomia = describir_anatomia(escenario, memoria)
for clave in ('motor_cognitivo', 'orquestador', 'system_message', 'memoria',
              'base_conocimiento', 'puerta_publicacion'):
    print(clave, json.dumps(anatomia[clave], ensure_ascii=False))
for h in anatomia['herramientas']:
    print('herramienta', h['nombre'], '|', h['papel'], '|', h['logica'], '|', h['argumentos'])

motor_cognitivo {"clase": "ChatOpenAI", "modelo": "gpt-4.1-mini", "disponible": false}
orquestador {"agente": "create_openai_tools_agent", "ejecutor": "AgentExecutor", "max_iteraciones": 6, "historial": "RunnableWithMessageHistory"}
system_message {"ptcf": {"Persona": true, "Tarea": true, "Contexto": true, "Formato": true}, "variables": ["agent_scratchpad", "input"], "variables_fijadas": ["chat_history", "dataset_id", "ejemplos", "formato"], "mensajes": ["SystemMessagePromptTemplate", "MessagesPlaceholder", "HumanMessagePromptTemplate", "MessagesPlaceholder"]}
memoria {"corto_plazo": "ventana de 12 mensajes + borrador por sesión", "largo_plazo": "preferencias explícitas + episodios vectoriales por usuario", "modelo_embeddings": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
base_conocimiento {"dataset_id": "e74ce1e8-f81c-468f-b674-4e6298b90e8f", "piezas": 2640, "fragmentos_politicas": 75, "origen_politicas": "ficticio_demo"}
puerta_publicacion {"verificacion": "exacta (

## R01 · agente híbrido: capa reactiva y capa deliberativa
Tres turnos sobre el mismo flujo final. **Doble de prueba:** el modelo es un guion determinista (no es OpenAI) y ViaCEP está simulado. Lo real es el flujo: las reglas, el `AgentExecutor`, la herramienta ejecutada y la verificación antes de publicar. Los dos primeros turnos los resuelve una regla sin pasar por el agente; en el segundo, el intérprete de turno lee el mensaje y la regla pregunta la cantidad. El tercero pasa por el agente, que recibe las seis herramientas y solicita una.

In [10]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Traza, crear_flujo
from pedro.memory import nueva_sesion
sesion = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
final = json.dumps({'estado': 'cep', 'dataset_id': escenario.dataset_id,
    'solicitud': {'cep': '01001000'}, 'cidade': 'Cidade Simulada', 'uf': 'SP',
    'modo_cep': 'simulado', 'evidencias': [{'tipo': 'viacep',
    'fuente_id': 'ViaCEP:01001000', 'texto': 'Cidade Simulada/SP'}]})
guion = [AIMessage(content='', tool_calls=[{'name': 'consultar_cep',
                                             'args': {'cep': '01001000'}, 'id': 'c1'}]),
         AIMessage(content=final)]
llamadas, lecturas = [], []
def modelo_doble(prompt, **kwargs):
    if 'tools' not in kwargs:  # lo llama el intérprete de turno, no el agente
        lecturas.append(1)
        return json.dumps({'es_compra': 'CEP' not in prompt.to_messages()[-1].content})
    llamadas.append(sorted(t['function']['name'] for t in kwargs['tools']))
    return guion[len(llamadas) - 1]
traza = Traza()
flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),
                    cep_simulado=True, traza=traza)
config = {'configurable': {'user_id': 'alumna', 'session_id': sesion}}
for pregunta in ('lista piezas', 'Compara PEC-01-0001',
                 '¿A qué ciudad corresponde el CEP 01001000?'):
    antes = (len(llamadas), len(traza.eventos), len(lecturas))
    respuesta = flujo.invoke({'input': pregunta}, config=config).content
    usadas = [e['herramienta'] for e in traza.eventos[antes[1]:] if e['evento'] == 'tool_start']
    n = len(llamadas) - antes[0]
    print('turno:', pregunta)
    print('  capa:', 'deliberativa' if n else 'reactiva', '| llamadas al agente:', n,
          '| lecturas del intérprete:', len(lecturas) - antes[2], '| herramientas ejecutadas:', usadas)
    print('  respuesta:', respuesta[:100].replace('\n', ' '))
print('herramientas ofrecidas al modelo:', llamadas[0])
assert len(llamadas) == 2 and [e['herramienta'] for e in traza.eventos
                               if e['evento'] == 'tool_start'] == ['consultar_cep']

turno: lista piezas
  capa: reactiva | llamadas al agente: 0 | lecturas del intérprete: 0 | herramientas ejecutadas: []
  respuesta: Piezas del escenario (2640; página 1): - PEC-01-0001 · NEVE-COMP-0001 · Compresor para Neveras NEVE-
turno: Compara PEC-01-0001
  capa: reactiva | llamadas al agente: 0 | lecturas del intérprete: 1 | herramientas ejecutadas: []
  respuesta: ¿Cuántas unidades de PEC-01-0001 necesitas? Puedo comparar sin presupuesto o plazo máximo si no tien
turno: ¿A qué ciudad corresponde el CEP 01001000?
  capa: deliberativa | llamadas al agente: 2 | lecturas del intérprete: 1 | herramientas ejecutadas: ['consultar_cep']
  respuesta: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
herramientas ofrecidas al modelo: ['buscar_productos', 'comparar_ofertas', 'consultar_catalogo', 'consultar_cep', 'consultar_politicas', 'guardar_preferencia']


/tmp/ipykernel_44590/2090706570.py:21: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),


Lectura: «lista piezas» y «Compara PEC-01-0001» se resuelven por regla (0 llamadas al agente); en la segunda el intérprete confirma que es una compra. La pregunta de CEP no tiene regla: el modelo elige `consultar_cep` entre seis herramientas y la respuesta solo se publica tras coincidir con la observación de la herramienta. Con un modelo real la elección la hace el LLM; esa ejecución remota corresponde a R26.

## R02 · planificación: una solicitud compleja dividida en pasos
Ejecución real local, sin modelo. La solicitud se divide en cinco subtareas, cada una con una salida observable: extraer restricciones, consultar ofertas, comparar, verificar y publicar.

In [11]:
from pedro.agent import renderizar
from pedro.business import comparar
from pedro.catalog import ofertas
from pedro.dialogue import interpretar_compra
from pedro.verification import respuesta_local, verificar
mensaje = 'Necesito 5 unidades de PEC-01-0001 hasta R$ 10.000 en 7 días'
estado, aclaracion = interpretar_compra(escenario, None, mensaje)
print('1 extraer  ', estado.solicitud.model_dump(exclude_none=True), '| falta:', aclaracion)
disponibles = ofertas(escenario, estado.solicitud.producto_id)
print('2 consultar', [o.id for o in disponibles])
alternativas = comparar(estado.solicitud, disponibles)
print('3 comparar ', [(a.oferta.proveedor_id, a.total_centavos, a.cumple, a.motivos)
                      for a in alternativas])
propuesta = respuesta_local(escenario, estado.solicitud)
print('4 verificar', 'errores:', verificar(propuesta, escenario))
print('5 publicar ', renderizar(propuesta).splitlines()[0])

1 extraer   {'producto_id': 'PEC-01-0001', 'cantidad': 5, 'presupuesto_centavos': 1000000, 'plazo_max_dias': 7} | falta: None
2 consultar ['PEC-01-0001-BR-05', 'PEC-01-0001-BR-06', 'PEC-01-0001-INT-06']
3 comparar  [('BR-06', 727983, True, []), ('BR-05', 193040, False, ['stock', 'plazo']), ('INT-06', 734492, False, ['plazo'])]
4 verificar errores: []
5 publicar  CentroPeças: R$ 7.279,83 para 5 unidades; plazo 2 días.


## R02 · memoria aumentada: qué va a corto y a largo plazo
Ejecución real local, sin modelo. Corto plazo: el borrador de compra y la ventana de 12 mensajes viven en la sesión. Largo plazo: la preferencia vive en el usuario y aparece en una sesión nueva; otro usuario no la ve. Los episodios vectoriales se demuestran en el notebook 03.

In [12]:
from pedro.memory import Historial, guardar_preferencia, leer_borrador, preferencias
sesion_a = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
local = crear_flujo(escenario, memoria)
def cfg(usuario, sesion):
    return {'configurable': {'user_id': usuario, 'session_id': sesion}}
for texto in ('Necesito 5 unidades de PEC-01-0001 en 7 días', 'ahora 6 unidades'):
    salida = local.invoke({'input': texto}, config=cfg('alumna', sesion_a)).content
    print(texto, '->', salida.splitlines()[0])
    print('  borrador (corto plazo):', {k: v for k, v in
          leer_borrador(memoria, 'alumna', sesion_a)['solicitud'].items() if v is not None})
for _ in range(7):
    local.invoke({'input': 'lista piezas'}, config=cfg('alumna', sesion_a))
historial = Historial(memoria, 'alumna', sesion_a)
print('mensajes guardados:', len(historial.todos()), '| ventana que ve el agente:', len(historial.messages))
guardar_preferencia(memoria, 'alumna', 'proveedor', 'nacional', explicita=True)
sesion_b = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
print('sesión nueva -> borrador:', leer_borrador(memoria, 'alumna', sesion_b),
      '| preferencia (largo plazo):', preferencias(memoria, 'alumna'),
      '| otro usuario:', preferencias(memoria, 'otro'))

/tmp/ipykernel_44590/1214989804.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  local = crear_flujo(escenario, memoria)


Necesito 5 unidades de PEC-01-0001 en 7 días -> CentroPeças: R$ 7.279,83 para 5 unidades; plazo 2 días.
  borrador (corto plazo): {'producto_id': 'PEC-01-0001', 'cantidad': 5, 'plazo_max_dias': 7}
ahora 6 unidades -> CentroPeças: R$ 8.718,75 para 6 unidades; plazo 2 días.
  borrador (corto plazo): {'producto_id': 'PEC-01-0001', 'cantidad': 6, 'plazo_max_dias': 7}


mensajes guardados: 18 | ventana que ve el agente: 12


sesión nueva -> borrador: None | preferencia (largo plazo): {'proveedor': 'nacional'} | otro usuario: {}


Lectura: al cambiar la cantidad a 6 el plazo de 7 días se conserva en el borrador. El historial guarda 18 mensajes pero el agente solo ve 12. La sesión nueva empieza sin borrador y conserva la preferencia del usuario.